# 5 · A price list: cardinality and validation

**The problem.** An online shop sells each product in several currencies. A product has *at most one* price per
currency, and a price object belongs to *exactly one* product and currency. Stock items belong to at most one
warehouse. Data arrives from spreadsheets and partner feeds, and it's often wrong in small ways: a price given as the
string `"12.99"`, two euro prices for the same product.

Two questions follow:

1. How do you *say* "at most one price per currency" in the schema?
2. When and how is data *checked* against it?

The answers: `unique(...)` clauses on relations, and a validator you call when you choose to.

In [1]:
from schemas.Framework import Proxies, Schemas, Validators

## A dictionary is a relation with a key

In Python you might write `product.prices: dict[str, Money]`. In this framework a map `product -> [currency] ->
price` is a relation with links `product` and `price` and a property `currency`. It's the same shape you've used
since case study 2. The key is simply an entry property.

In [2]:
Product = Schemas.OfObject.Builder().properties(lambda prop: prop.name('sku').of(lambda t: t.as_native(str))).create()
Money = Schemas.OfObject.Builder().properties(lambda prop: prop.name('minor_units').of(lambda t: t.as_native(int))).create()  # cents

Prices = (
    Schemas.OfRelation.Builder()
    .links('product', 'price')
    .properties(lambda prop: prop.name('currency').of(lambda t: t.as_native(str)))  # ISO 4217, e.g. 'EUR'
    .unique('price')                  # (product, currency) determine the price: one price per currency
    .unique('product', 'currency')    # the price determines (product, currency): a price is used once
    .create()
)
Product = Schemas.OfObject.Builder(Product).relations(lambda adj: adj.name('prices').of(Prices).me('product')).update()
Money = Schemas.OfObject.Builder(Money).relations(lambda adj: adj.name('priced').of(Prices).me('price')).update()

## Reading `unique(S)`

`unique(S)` names the things that are *determined*. Everything **not** in `S`, taken together, is a key: two entries
that agree on everything outside `S` must also agree on `S`.

| Clause on `Prices` | The key (everything else) | In words |
|---|---|---|
| `unique('price')` | `product`, `currency` | a product has at most one price per currency |
| `unique('product', 'currency')` | `price` | a price object is used by at most one (product, currency) |

**Why this notation, and not "one-to-many" labels?** Real relations have more than two ends and carry properties.
"One-to-many" can't express "one price per product *per currency*". Naming what is determined by the rest covers
every case with one idea, and you add a clause per rule. The common patterns:

| Pattern | Declaration |
|---|---|
| Ownership: each child has at most one parent | `.links('parent', 'child').unique('parent')` |
| Keyed ownership: one parent and one key per child | `.links('parent', 'child').properties(key).unique('parent', 'key')` |
| Directory: also, each key names at most one child per parent | ... `.unique('parent', 'key').unique('child')` |
| At most one entry in the whole relation | `unique` over every link and property |

`unique` gives only an upper bound. "At least one" is a different kind of rule, which the framework doesn't
express yet. Nothing is mandatory until a constraint says so, as in case study 1.

In [3]:
Warehouse = Schemas.OfObject.Builder().properties(lambda prop: prop.name('city').of(lambda t: t.as_native(str))).create()
Stocking = Schemas.OfRelation.Builder().links('warehouse', 'product').unique('warehouse').create()  # ownership
Warehouse = Schemas.OfObject.Builder(Warehouse).relations(lambda adj: adj.name('stock').of(Stocking).me('warehouse')).update()
Product = Schemas.OfObject.Builder(Product).relations(lambda adj: adj.name('stored_in').of(Stocking).me('product')).update()

for name, schema in [('Product', Product), ('Money', Money), ('Prices', Prices),
                     ('Warehouse', Warehouse), ('Stocking', Stocking)]:
    assert schema.validate() == [], (name, schema.validate())
    Proxies.register(name, schema)
Builders = Proxies.Builders

`Schema.validate()` also checks the clauses themselves. For example, `unique` may only name links and properties the
relation has:

In [4]:
Schemas.OfRelation.Builder().links('a', 'b').unique('c').create().validate()

["unique(c) names unknown links or properties ['c']"]

## Good data

A validator is constructed with a *registry*, which it uses to look up schemas by the names objects carry.
`Proxies.Builders` is that registry for proxies. Call it with a schema and a value. It returns a list of problems,
empty when all is well:

In [5]:
validate = Validators.Validate(Proxies.Builders)

widget = (
    Builders.Product()
    .sku('W-1')
    .prices(lambda x: x.price(lambda m: m.minor_units(1299)).currency('EUR'))
    .prices(lambda x: x.price(lambda m: m.minor_units(1500)).currency('JPY'))
    .create()
)
validate(Product, widget)

[]

## Bad data, all of it at once

A partner feed sends a gadget with two euro prices, one of which was typed as a string:

In [6]:
gadget = (
    Builders.Product()
    .sku('G-1')
    .prices(lambda x: x.price(lambda m: m.minor_units(900)).currency('EUR'))
    .prices(lambda x: x.price(lambda m: m.minor_units('950')).currency('EUR'))
    .create()
)
for problem in validate.Reachable(Product, gadget):
    print(problem)

Money#2.minor_units: expected int, got str
unique(price) violated: entries agreeing on ['currency', 'product'] differ on ['price']


Each problem says where it is. `Money#2.minor_units` is the `minor_units` of the object numbered 2 in the order the
validator reached them, and a type problem inside an entry is written as a path like `Product#0.prices[1].currency`.
The uniqueness message names the key the entries agree on and what they disagree about.

`validate(schema, value)` checks the value and its entries. `validate.Reachable(schema, root)` checks everything
reachable from the root, as defined in case study 4. That's how the bad `Money` object was found.

### Why doesn't the builder stop this?

Case study 1 showed that builders accept any value. Now the reasons are concrete:

- **You get every problem, not the first.** A feed with 2,000 errors produces a report, not 2,000 fix-and-rerun
  cycles.
- **Intermediate states are legitimate.** The gadget briefly had two euro prices, and a later edit in the same
  import could remove one. Rules that can only be judged on the finished data can't be enforced at every step.
- **You choose when to pay for it.** Checking uniqueness means looking at every entry. Doing it on every `update()`
  would make bulk loads quadratic.
- **Some checks span many objects.** Uniqueness is about entries across objects, not a single setter call.

The trade-off: data is only known to be valid after you've validated it. Validate at trust boundaries: after an
import, before a save, before handing data to another system.

## Fixing it

Using case study 3's removal, drop the entry whose price is the string:

In [7]:
from toolkit import remove_entries

remove_entries(gadget, 'prices', where=lambda row: isinstance(row['price'].minor_units, str))
validate.Reachable(Product, gadget)

[]

## Links must be filled by the right kind of object

Links are untyped in the relation, but the *objects* declare which links they can fill. The validator checks each
linked object's schema against that. Here a `Warehouse` has been linked where a price belongs. The builder allowed
it, because an existing object can always be linked, and the validator catches it:

In [8]:
berlin = Builders.Warehouse().city('Berlin').create()
Builders.Product(widget).prices(lambda x: x.price(berlin).currency('GBP')).update()
for problem in validate(Product, widget):
    print(problem)
widget = remove_entries(widget, 'prices', where=lambda row: row['price'] is berlin)  # returns the updated object

Product#0.prices[2].price: a 'Warehouse' cannot fill link 'price'; its schema declares no adjacency to this relation via 'price'


## Ownership

`Stocking` says a product is in at most one warehouse. Stocking the widget in two breaks it:

In [9]:
paris = Builders.Warehouse().city('Paris').create()
Builders.Warehouse(berlin).stock(lambda x: x.product(widget)).update()
Builders.Warehouse(paris).stock(lambda x: x.product(widget)).update()
validate.Reachable(Product, widget)

["unique(warehouse) violated: entries agreeing on ['product'] differ on ['warehouse']"]

Uniqueness is checked over the entries the validator sees, which with `Reachable` is the whole connected component.
(Entries in a separate component that happen to share property values are not compared. That's an open design
question, recorded as finding F7.)

## Validating a single value

The validator works for any schema, not just objects. This is handy in form handlers:

In [10]:
print(validate(Schemas.OfNative.Data(int), 3), validate(Schemas.OfNative.Data(int), True))

[] ['expected int, got bool']


`True` isn't an `int` here, even though Python says `isinstance(True, int)`. Native types are exact and never coerced
into each other, in validation, in serialization and in equality. `1`, `1.0` and `True` are three different values.
That's what lets the same data mean the same thing in a language where they really are different types.

## What you learned

- Maps are relations with a key property. `unique(S)` says the rest of the entry determines `S`.
- Ownership, keyed ownership and directories are combinations of `unique` clauses.
- `Validators.Validate(Proxies.Builders)` checks data only when called, and returns every problem with a path.
  `.Reachable` checks a whole component.
- Linked objects must have a schema that declares the link they fill.
- Native types are exact: `True` is not an `int`, and `1.0` is not `1`.

**Next:** the data is right. Case study 6 saves it and loads it back, which is where the insistence on relations,
schema names and exact types pays off.